In [1]:
!pip install python-telegram-bot==20.7 nest-asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 552.6/552.6 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 4.1 MB/s eta 0:00:00
  Attempting uninstall: httpx
    Found existing installation: httpx 0.28.1
    Uninstalling httpx-0.28.1:
      Successfully uninstalled httpx-0.28.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires httpx<1.0.0,>=0.27.0, but you have httpx 0.25.2 which is incompatible.
mcp 1.27.0 requires httpx>=0.27.1, but you have httpx 0.25.2 which is incompatible.
google-genai 1.68.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.25.2 which is incompatible.
firebase-admin 6.9.0 requires httpx[http2]==0.28.1, but you have httpx 0.25.2 which is incompatible.


In [2]:
import logging
import nest_asyncio
import asyncio
from telegram import Update, ReplyKeyboardMarkup, ReplyKeyboardRemove
from telegram.ext import Application, CommandHandler, MessageHandler, filters, ConversationHandler, ContextTypes

In [3]:
nest_asyncio.apply()
TOKEN = '8718579709:AAGjBJacPxzMcxIqMZFbLg8np3cgS2k0E0c'

In [4]:
logging.basicConfig(format='%(asctime)s - %(name)s - %(levelname)s - %(message)s', level=logging.INFO)

WAITING_FOR_DESCRIPTION = 1


user_memes = {}

async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Приветственное сообщение"""
    await update.message.reply_text(
        "🤖 *Привет! Я бот-мемогенератор*\n\n"
        "📋 *Доступные команды:*\n"
        "/creatememe - создать новый мем\n"
        "/help - список команд\n"
        "/cancel - отменить создание\n"
        "/status - статус генерации\n\n"
        "❌ Если напишешь ерунду - покажу список команд\n\n"
        "🎮 *Погнали!*",
        parse_mode="Markdown"
    )

async def help_command(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Помощь"""
    await update.message.reply_text(
        "📖 *Справочная информация*\n\n"
        "🔹 */creatememe* - запустить создание мема\n"
        "🔹 */cancel* - отменить текущее действие\n"
        "🔹 */status* - проверить статус бота\n"
        "🔹 */start* - приветствие\n\n"
        "*Как создать мем:*\n"
        "1️⃣ Введи /creatememe\n"
        "2️⃣ Опиши любой мем текстом\n"
        "3️⃣ Бот покажет результат\n\n",
        parse_mode="Markdown"
    )

async def status(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Проверка статуса бота"""
    await update.message.reply_text(
        "✅ *Бот работает в штатном режиме*\n"
        "📊 uptime: активен",
        parse_mode="Markdown"
    )

async def cancel(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Отмена создания мема"""
    if update.effective_user.id in user_memes:
        del user_memes[update.effective_user.id]

    await update.message.reply_text(
        "❌ *Действие отменено*\n\n"
        "Чтобы создать новый мем - введи /creatememe",
        parse_mode="Markdown",
        reply_markup=ReplyKeyboardRemove()
    )
    return ConversationHandler.END

async def creatememe_start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Начало создания мема - запрос описания"""
    await update.message.reply_text(
        "🎨 *Опиши мем и я сгенерирую его*\n\n"
        "✏️ *Примеры описаний:*\n"
        "• «кот грустит из-за дождя»\n"
        "• «человек-паук опаздывает на работу»\n"
        "• «пельмень мечтает стать вареником»\n"
        "• «собака пытается понять физику»\n\n"
        "💬 *Напиши своё описание:*\n"
        "(Чтобы отменить - введи /cancel)",
        parse_mode="Markdown"
    )
    return WAITING_FOR_DESCRIPTION

async def process_meme_description(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Обработка описания и генерация (пока заглушка)"""
    user_id = update.effective_user.id
    user_description = update.message.text.strip()
    user_memes[user_id] = user_description

    thinking_msg = await update.message.reply_text("🎨 *Генерирую мем...*\n(пока тестовый режим)", parse_mode="Markdown")
    await asyncio.sleep(1.5) #  как будто нейросеть работает
    # тут будем генерировать мем, а пока что проверка чётности длины стоит
    description_length = len(user_description)

    if description_length % 2 == 0:
        result = "0️⃣"
        parity = "чётная"
        emoji = "✅"
    else:
        result = "1️⃣"
        parity = "нечётная"
        emoji = "🎲"

    await thinking_msg.delete()

    await update.message.reply_text(
        f"{emoji} *Результат генерации* \n\n"
        f"📝 *Твоё описание:*\n«{user_description}»\n\n"
        f"🔢 *Длина описания:* {description_length} символов ({parity})\n"
        f"🎯 *Тестовый вывод:* {result}\n\n"
        "✨ *Что дальше?*\n"
        "• Хочешь создать ещё один мем? Введи /creatememe\n"
        "• Нужна помощь? Введи /help\n\n"
        "🚀 *Скоро здесь будет реальная нейросеть!*",
        parse_mode="Markdown"
    )

    if user_id in user_memes:
        del user_memes[user_id]

    return ConversationHandler.END

async def handle_random_text(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Обработка случайного текста (не команды)"""
    await update.message.reply_text(
        "❓ *Я тебя не понял*\n\n"
        "Вот что я умею:\n"
        "🎨 /creatememe - создать мем\n"
        "📖 /help - все команды\n"
        "❌ /cancel - отменить действие\n\n"
        "Просто напиши одну из команд 👆",
        parse_mode="Markdown"
    )

async def post_init(application: Application):
    """Действия после инициализации бота"""
    print("✅ Бот успешно запущен!")
    print("💡 Команды: /start, /creatememe, /help, /cancel, /status")

def main():
    """Запуск бота"""
    app = Application.builder().token(TOKEN).post_init(post_init).build()
    conv_handler = ConversationHandler(
        entry_points=[CommandHandler('creatememe', creatememe_start)],
        states={
            WAITING_FOR_DESCRIPTION: [
                MessageHandler(filters.TEXT & ~filters.COMMAND, process_meme_description)
            ],
        },
        fallbacks=[CommandHandler('cancel', cancel)],
        name="meme_creation",
        persistent=False
    )

    app.add_handler(conv_handler)
    app.add_handler(CommandHandler("start", start))
    app.add_handler(CommandHandler("help", help_command))
    app.add_handler(CommandHandler("status", status))
    app.add_handler(CommandHandler("cancel", cancel))
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_random_text))

    try:
        loop = asyncio.get_running_loop()
        loop.create_task(app.run_polling())
    except RuntimeError:
        app.run_polling()

if __name__ == "__main__":
    main()

✅ Бот успешно запущен!
💡 Команды: /start, /creatememe, /help, /cancel, /status
✅ Бот успешно запущен!
💡 Команды: /start, /creatememe, /help, /cancel, /status


RuntimeError: Cannot close a running event loop